# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring the FAIR^2 clinical dataset using the `mlcroissant` library, employing Croissant schema entity `@id` references for all data access and processing steps.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Below, we summarize the available record sets, their fields, and the corresponding Croissant schema `@id`s. This information is crucial for programmatically referencing dataset structures.

In [ ]:
# Utility to print RecordSets and Fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field name: {field.name}")
        print(f"      @id: {field.id}")
        if hasattr(field, 'data_type'):
            print(f"      dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load all available record sets into pandas DataFrames using their `@id`s. The mapping between record set names and Croissant schema `@id`s is retained for easy reference.

Let's list the record set `@id`s available in this dataset, and load all (since the main data table contains clinical records of 77 cancer survivors).

In [ ]:
# Gather all record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @id list:")
for i, rset_id in enumerate(record_set_ids):
    print(f"  [{i}] {rset_id}")

# Load all record sets into dataframes, by @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} with shape {df.shape}")

# Show first few columns of main table (assume first record set is the main data)
main_record_set_id = record_set_ids[0]
print(f"\nColumns in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps to prepare the data for further analysis. 

- We start by identifying a numeric field (by its `@id`) to illustrate outlier removal and normalization.
- Then, we explore grouping and aggregation operations using a categorical field.
- All field references use Croissant `@id`.

In [ ]:
# --- Custom setup: Identify a numeric field and a grouping field ---
# By inspecting field data types in the overview above, suppose the field representing age at diagnosis is:
# Numeric field example: '@id': 'http://senscience.ai/field/age_at_second_crc_diagnosis'
numeric_field_id = 'http://senscience.ai/field/age_at_second_crc_diagnosis'
# For grouping: say we want to group by MSI status (@id):
group_field_id = 'http://senscience.ai/field/msi_status'

# Set up df
df = dataframes[main_record_set_id]

# Filter rows (ages > 55 for example)
min_age = 55
filtered_df = df[df[numeric_field_id] > min_age]
print(f"Filtered records where {numeric_field_id} > {min_age} (N={len(filtered_df)}):")
display_cols = [numeric_field_id]
if group_field_id in filtered_df.columns:
    display_cols.append(group_field_id)
print(filtered_df[display_cols].head())

# Normalize numeric field in-place
col_norm = numeric_field_id + '_normalized'
filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print("\nNormalized age_at_second_crc_diagnosis for filtered records:")
print(filtered_df[[numeric_field_id, col_norm]].head())

# Group by MSI status and compute mean/summary
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count','mean','std']).reset_index()
    print(f"\nGrouped statistics by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Let's visualize the distribution of age at second CRC diagnosis, colored by MSI status. All field references use Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize age at diagnosis (by MSI status if available)
plt.figure(figsize=(8,5))
if group_field_id in df.columns and df[group_field_id].nunique() > 1:
    sns.histplot(data=df, x=numeric_field_id, hue=group_field_id, kde=True, bins=15, alpha=0.7)
    plt.title('Distribution of Age at Second CRC Diagnosis by MSI Status')
    plt.xlabel(numeric_field_id)
    plt.legend(title=group_field_id)
else:
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title('Distribution of Age at Second CRC Diagnosis')
    plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 6. Conclusion

- Successfully loaded clinical dataset defined by the Croissant schema, referencing all data elements by their unique `@id`.
- Extracted and processed clinical records, illustrated primary field selection and normalization, and examined relationships between clinicopathological fields.
- Visualized key field distributions. This approach using `mlcroissant` and Croissant `@id` referencing allows robust, reproducible data science workflows across FAIR datasets.